In [22]:
%pip install ultralytics mlflow opencv-python-headless pyyaml pandas matplotlib scikit-learn kagglehub imutils

Note: you may need to restart the kernel to use updated packages.


In [23]:
# Cellule 2 : Importations
import os
import glob
import yaml
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import mlflow
import kagglehub
from imutils import paths
from ultralytics import YOLO


In [24]:
# Download latest version
path = kagglehub.dataset_download("ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes")

print("Path to dataset files:", path)

image_paths = list(paths.list_images(path))


Path to dataset files: /root/.cache/kagglehub/datasets/ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes/versions/1


In [25]:
def check_dataset(image_dir, label_dir):
    valid_ext = ('.jpeg', '.jpg', '.png', '.bmp')
    images = [f for f in os.listdir(image_dir) if f.lower().endswith(valid_ext)]
    labels = [f for f in os.listdir(label_dir) if f.endswith('.txt')]

    img_basenames = set([os.path.splitext(img)[0] for img in images])
    lbl_basenames = set([os.path.splitext(lbl)[0] for lbl in labels])

    orphans_img = img_basenames - lbl_basenames
    orphans_lbl = lbl_basenames - img_basenames

    print(f"Directory {image_dir}:")
    print(f"  - Orphan images: {len(orphans_img)}")
    print(f"  - Orphan annotations: {len(orphans_lbl)}")
    return orphans_img, orphans_lbl



In [26]:

data_yaml_path = "data.yaml"

train_path = 'Train'
val_path = 'Val'

data_config = {
    'path': path,
    'train': train_path,
    'val': val_path,
    'test': val_path,
    'nc': 1,
    'names': ['tumor']
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Updated data.yaml generated successfully.")
print(f"Train path mapped to: {train_path}")
print(f"Val/Test path mapped to: {val_path}")

Updated data.yaml generated successfully.
Train path mapped to: Train
Val/Test path mapped to: Val


In [27]:
print("Folders in dataset root:", os.listdir(path))

Folders in dataset root: ['Train', 'Val']


In [28]:
# Cell 6: YOLOv8 Training (Local Only)
# Load a pre-trained YOLOv8 model (Small model chosen for a balance of speed and accuracy)
model = YOLO('yolov8s.pt')

# Start training (Results will be saved locally in NeuroVision/yolov8s_run1)
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,          # Adjust based on your GPU VRAM
    patience=10,       # Early stopping to prevent overfitting
    project='NeuroVision',
    name='yolov8s_run1',
    # Requested augmentation parameters:
    degrees=10.0,      # Slight rotations
    translate=0.05,    # Translation
    scale=0.1,         # Scaling
    fliplr=0.0,        # Disabled: no horizontal flipping for brains
    mosaic=0.5         # Mosaic setting
)

Ultralytics 8.4.192 🚀 Python-3.10.22 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 24163MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolov8s_run1-4, nbs=64, nms=None, opset=None, optimize=False, op